In [2]:
%%bigquery magic
CREATE SCHEMA IF NOT EXISTS emergency_lab;

Query is running:   0%|          |

In [5]:
%%bigquery
CREATE SCHEMA IF NOT EXISTS emergency_lab;

Query is running:   0%|          |

""


In [7]:
%%bigquery

SELECT schema_name
FROM `region-us`.INFORMATION_SCHEMA.SCHEMATA
WHERE schema_name = 'emergency_lab';


Query is running:   0%|          |

Downloading:   0%|          |

,schema_name
0,emergency_lab


In [14]:
%%bigquery

CREATE OR REPLACE TABLE emergency_lab.emergency_calls_raw (
  call_type STRING,
  priority STRING,
  units_dispatched INT64,
  distance_miles FLOAT64,
  area_type STRING,
  response_time FLOAT64
);

LOAD DATA OVERWRITE emergency_lab.emergency_calls_raw
FROM FILES (
  format = 'CSV',
  uris = ['gs://labs.roitraining.com/data-to-ai-workshop/emergency_calls_response_times.csv'],
  skip_leading_rows = 1
);

Query is running:   0%|          |

""


In [15]:
%%bigquery
SELECT *
FROM emergency_lab.emergency_calls_raw
LIMIT 10;

Query is running:   0%|          |

Downloading:   0%|          |

,call_id,call_timestamp,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available,response_time
0,35957,2023-01-01 00:05:53+00:00,Fire,Highland,Rainy,Sunday,0,High,21.45,3,23.41
1,20832,2023-01-01 00:20:47+00:00,Fire,Oakmont,Rainy,Sunday,0,High,22.29,6,20.11
2,27949,2023-01-01 00:33:27+00:00,Fire,Riverside,Windy,Sunday,0,High,17.19,14,19.75
3,20199,2023-01-01 00:48:29+00:00,Fire,Riverside,Windy,Sunday,0,High,17.39,14,20.76
4,46938,2023-01-01 00:50:44+00:00,Rescue,Brookfield,Sunny,Sunday,0,High,22.50,14,22.37
5,17582,2023-01-01 02:28:50+00:00,Rescue,Downtown,Snowy,Sunday,2,High,25.15,6,28.48
6,21624,2023-01-01 02:44:06+00:00,Rescue,Oakmont,Snowy,Sunday,2,High,3.95,9,19.30
7,36793,2023-01-01 02:53:54+00:00,Fire,Riverside,Sunny,Sunday,2,High,5.87,10,10.72
8,41350,2023-01-01 03:52:33+00:00,Police,Greenfield,Windy,Sunday,3,High,6.66,5,20.55
9,32092,2023-01-01 04:09:23+00:00,Police,Maplewood,Snowy,Sunday,4,High,15.50,13,22.98


In [16]:
%%bigquery
SELECT column_name, data_type
FROM emergency_lab.INFORMATION_SCHEMA.COLUMNS
WHERE table_name = 'emergency_calls_raw'
ORDER BY ordinal_position;

Query is running:   0%|          |

Downloading:   0%|          |

,column_name,data_type
0,call_id,INT64
1,call_timestamp,TIMESTAMP
2,call_type,STRING
3,location,STRING
4,weather_condition,STRING
5,day_of_week,STRING
6,time_of_day,INT64
7,traffic_level,STRING
8,distance_to_station,FLOAT64
9,units_available,INT64


In [17]:
%%bigquery
SELECT
  COUNT(*) AS total_calls,
  AVG(response_time) AS avg_response_time,
  MIN(response_time) AS min_response_time,
  MAX(response_time) AS max_response_time
FROM emergency_lab.emergency_calls_raw;

Query is running:   0%|          |

Downloading:   0%|          |

,total_calls,avg_response_time,min_response_time,max_response_time
0,50000,17.446134,2.01,36.55


In [18]:
%%bigquery
CREATE OR REPLACE MODEL emergency_lab.response_time_model
OPTIONS (
  model_type = 'linear_reg',
  input_label_cols = ['response_time']
) AS
SELECT
  *
FROM emergency_lab.emergency_calls_raw
WHERE response_time IS NOT NULL;

Query is running:   0%|          |

""


In [19]:
%%bigquery
SELECT *
FROM ML.EVALUATE(
  MODEL emergency_lab.response_time_model,
  (
    SELECT *
    FROM emergency_lab.emergency_calls_raw
    WHERE response_time IS NOT NULL
  )
);

Query is running:   0%|          |

Downloading:   0%|          |

,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,1.739457,4.727395,0.0147,1.478236,0.831504,0.831505


In [22]:
%%bigquery
SELECT column_name, data_type
FROM emergency_lab.INFORMATION_SCHEMA.COLUMNS
WHERE table_name = 'emergency_calls_raw'
ORDER BY ordinal_position;

Query is running:   0%|          |

Downloading:   0%|          |

,column_name,data_type
0,call_id,INT64
1,call_timestamp,TIMESTAMP
2,call_type,STRING
3,location,STRING
4,weather_condition,STRING
5,day_of_week,STRING
6,time_of_day,INT64
7,traffic_level,STRING
8,distance_to_station,FLOAT64
9,units_available,INT64


In [25]:
%%bigquery
CREATE OR REPLACE MODEL emergency_lab.response_time_model
OPTIONS (
  model_type = 'linear_reg',
  input_label_cols = ['response_time']
) AS
SELECT
  call_type,
  location,
  weather_condition,
  day_of_week,
  time_of_day,
  traffic_level,
  distance_to_station,
  units_available,
  response_time
FROM emergency_lab.emergency_calls_raw
WHERE response_time IS NOT NULL;

Query is running:   0%|          |

""


In [26]:
%%bigquery
SELECT *
FROM ML.EVALUATE(
  MODEL emergency_lab.response_time_model,
  (
    SELECT
      call_type,
      location,
      weather_condition,
      day_of_week,
      time_of_day,
      traffic_level,
      distance_to_station,
      units_available,
      response_time
    FROM emergency_lab.emergency_calls_raw
    WHERE response_time IS NOT NULL
  )
);

Query is running:   0%|          |

Downloading:   0%|          |

,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,1.741854,4.73939,0.014721,1.479472,0.831076,0.831078


In [27]:
%%bigquery
SELECT *
FROM ML.PREDICT(
  MODEL emergency_lab.response_time_model,
  (
    SELECT
      'MEDICAL'      AS call_type,
      'DOWNTOWN'     AS location,
      'RAIN'         AS weather_condition,
      'MONDAY'       AS day_of_week,
      14             AS time_of_day,
      'HIGH'         AS traffic_level,
      4.2            AS distance_to_station,
      3              AS units_available
  )
);

Query is running:   0%|          |

Downloading:   0%|          |

,predicted_response_time,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available
0,1072.170892,MEDICAL,DOWNTOWN,RAIN,MONDAY,14,HIGH,4.2,3
